In [ ]:
# In[1]:

import json
from datetime import datetime
import pytz

# Load the task.json file
task_directory = 'rca100/cases/t053'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

# Convert alert window timestamps to UTC+8 timezone
timezone = pytz.timezone('Asia/Shanghai')
alert_start = datetime.fromisoformat(alert_window['start']).astimezone(timezone)
alert_end = datetime.fromisoformat(alert_window['end']).astimezone(timezone)

alert_title, alert_start, alert_end, alert_entity

```
Out[1]:
```
The alert is titled "inventory接口响应超时告警" (inventory interface response timeout alert). It occurred between 16:01:41 and 16:06:48 on April 3, 2026 (UTC+8). The alert was triggered by the APM operation `inventory::/api/v1/inventory/batch-reconciliation`.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口响应超时告警', datetime.datetime(2026, 4, 3, 16, 1, 41, 792896, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), datetime.datetime(2026, 4, 3, 16, 6, 48, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), {'entity_id': '2d5ef468202802748423656575955265', 'entity_name': 'inventory::/api/v1/inventory/batch-reconciliation', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all unique KPIs and their associated entity_set values
unique_kpis = metrics_data[['metric', 'entity_set']].drop_duplicates()

unique_kpis

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various `entity_set` values. Examples include `node_ready_status` and `node_memory_usage_rate` under `k8s.node`, as well as `error_rate` under both `apm.operation` and `apm.instance`. For a complete list of KPIs, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
540     node_not_ready_status            k8s.node
4050   node_memory_usage_rate            k8s.node
5940   node_pod_running_count            k8s.node
7695     node_disk_usage_rate            k8s.node
...                       ...                 ...
75425                workload  apm.service.legacy
75431                   error  apm.service.legacy
75434                    slow  apm.service.legacy
85412              error_rate       apm.operation
85581              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data for relevant components
relevant_components = ['inventory', 'frontend', 'checkout', 'payment', 'cart', 'product-catalog', 
                        'accounting', 'shipping', 'email', 'recommendation', 'currency', 'ad', 
                        'fraud-detection', 'image-provider', 'quote', 'load-generator', 'flagd']

filtered_metrics = metrics_data[metrics_data['entity_name'].str.contains('|'.join(relevant_components), case=False, na=False)]

# Ensure 'value' column is numeric for aggregation
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Calculate global P95 threshold for each 'component-KPI' time series
filtered_metrics['component'] = filtered_metrics['entity_name'].str.split('::').str[0]  # Extract component name
p95_thresholds = (
    filtered_metrics.groupby(['component', 'metric'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

p95_thresholds

```
Out[3]:
```
The filtered metrics data includes KPIs for relevant components such as `inventory`, `frontend`, `checkout`, and others. For each component-KPI combination, the global P95 threshold has been calculated. For example, the `accounting` component has a P95 threshold of 0.0508 seconds for `avg_request_latency_seconds` and 4,771 requests for `request_count`. The full table contains 216 rows, and you can request to display more rows for a comprehensive view.

The original code execution output of IPython Kernel is also provided below for reference:

component                       metric  p95_threshold
0    accounting  avg_request_latency_seconds   5.077629e-02
1    accounting                          cpu   6.894275e-03
2    accounting                      latency   4.947270e-02
3    accounting                          mem   1.051143e+08
4    accounting                request_count   4.771100e+03
..          ...                          ...            ...
211    shipping                          cpu   1.200511e-02
212    shipping                      latency   1.283294e-03
213    shipping                          mem   3.194334e+06
214    shipping                request_count   9.767400e+03
215    shipping                     workload   1.048500e+04

[216 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Convert the 'time' column to numeric and filter data within the alert time window
filtered_metrics['time'] = pd.to_numeric(filtered_metrics['time'], errors='coerce')
alert_start_microseconds = int(alert_start.timestamp() * 1_000_000)
alert_end_microseconds = int(alert_end.timestamp() * 1_000_000)

metrics_within_window = filtered_metrics[
    (filtered_metrics['time'] >= alert_start_microseconds) &
    (filtered_metrics['time'] <= alert_end_microseconds)
]

# Merge with P95 thresholds to compare values
metrics_with_thresholds = metrics_within_window.merge(
    p95_thresholds,
    on=['component', 'metric'],
    how='left'
)

# Identify anomalies where the value exceeds the P95 threshold
anomalies = metrics_with_thresholds[metrics_with_thresholds['value'] > metrics_with_thresholds['p95_threshold']]

anomalies

```
Out[4]:
```
Within the alert time window (16:01:41 to 16:06:48, UTC+8), 128 anomalies were identified where the values of specific component-KPI combinations exceeded their respective global P95 thresholds. Examples include the `frontend-proxy::ingress` operation with a value of 59,617 exceeding its P95 threshold of 59,005, and the `frontend::POST` operation with a value of 59,581 exceeding its P95 threshold of 55,486.7. The anomalies span multiple components, including `frontend`, `frontend-proxy`, and `inventory`. For a complete view, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                      entity_name  ...         value                            metric_set_id service                        component p95_threshold
11    1775203601000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  ...  59617.000000  apm@metric_set@apm.metric.apm.operation    None                   frontend-proxy  59005.000000
29    1775203601000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098                   frontend::POST  ...  59581.000000  apm@metric_set@apm.metric.apm.operation    None                         frontend  55486.700000
54    1775203541000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  ...   2233.000000  apm@metric_set@apm.metric.apm.operation    None                   frontend-proxy   2221.400000
83    1775203541000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098                   frontend::POST  ...   2266.000000  apm@metric_set@apm.metric.apm.operation    None                         frontend   1483.600000
145   1775203601000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc          frontend-proxy::ingress  ...  61964.000000  apm@metric_set@apm.metric.apm.operation    None                   frontend-proxy  58044.900000
...                ...    ...            ...                               ...                              ...  ...           ...                                      ...     ...                              ...           ...
2399  1775203601000000    apm   apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd  ...      0.959904   apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-xz2fd      0.946232
2411  1775203601000000    apm   apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b  ...      0.971930   apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-8c99b      0.956966
2417  1775203601000000    apm   apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd        frontend-6df7d557f5-sgfcb  ...      0.964745   apm@metric_set@apm.metric.apm.instance    None        frontend-6df7d557f5-sgfcb      0.950812
2419  1775203511000000    apm   apm.instance  7fb113e159d1fa7ccdfc641acf1870ad       inventory-794d9f8f85-hhjtj  ...      0.012000   apm@metric_set@apm.metric.apm.instance    None       inventory-794d9f8f85-hhjtj      0.011512
2425  1775203601000000    apm   apm.instance  beecfe3febef4ed2425bac5626962ed9        frontend-6df7d557f5-nz7g4  ...      0.962583   apm@metric_set@apm.metric.apm.instance    None        frontend-6df7d557f5-nz7g4      0.944012

[128 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' to numeric and filter data within the alert time window
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

alert_start_nanoseconds = alert_start_microseconds * 1_000  # Convert to nanoseconds
alert_end_nanoseconds = alert_end_microseconds * 1_000  # Convert to nanoseconds

traces_within_window = traces_data[
    (traces_data['startTime'] >= alert_start_nanoseconds) &
    (traces_data['endTime'] <= alert_end_nanoseconds)
]

# Filter traces involving the 'inventory' service and other faulty components
faulty_components = ['inventory', 'frontend', 'frontend-proxy']
traces_faulty_components = traces_within_window[
    traces_within_window['serviceName'].str.contains('|'.join(faulty_components), case=False, na=False)
]

# Identify the most downstream faulty service in the call chain
# Sort by 'startTime' to analyze the sequence of calls
traces_faulty_components_sorted = traces_faulty_components.sort_values(by='startTime')

traces_faulty_components_sorted

```
Out[5]:
```
The filtered trace data within the alert time window includes 29,714 traces involving the `inventory` service and other faulty components like `frontend` and `frontend-proxy`. These traces reveal the sequence of calls, such as ingress, router egress, and specific API routes like `/api/v1/inventory/batch-reconciliation`. To determine the most downstream faulty service in the call chain, further analysis of the trace hierarchy and parent-child relationships is required. For a detailed view, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                                           spanName  ...  __topic__      __tag__:__pack_id__ events links         ip
400347  d459e664e6b4af1c0870f345edefecf4  c1b628526503e8a4                      2                                            ingress  ...             6B168B6371B6AC7E-381CEB   None  None       None
400346  d459e664e6b4af1c0870f345edefecf4  80e00f9434ad570b  c1b628526503e8a4    3                             router frontend egress  ...             6B168B6371B6AC7E-381CEB   None  None       None
400537  d459e664e6b4af1c0870f345edefecf4  93d7928f54346dad  80e00f9434ad570b    2                                                GET  ...             3AAB798034CD4509-37F9AA   None  None       None
400520  d459e664e6b4af1c0870f345edefecf4  5b88df84b7b1f5e5  8717595e1ee7fa85    3     grpc.oteldemo.ProductCatalogService/GetProduct  ...             3AAB798034CD4509-37F9AA   None  None       None
400528  d459e664e6b4af1c0870f345edefecf4  f32e0669db527cf8  93d7928f54346dad    2                      GET /api/products/{productId}  ...             3AAB798034CD4509-37F9AA   None  None       None
...                                  ...               ...               ...  ...                                                ...  ...        ...                      ...    ...   ...        ...
460001  cd04d56d7eaee6c17d749ed6658f8cf3  e7f1e15618e10088  6e76695156df2f7e    3                                         dns.lookup  ...             F5B9811ED13C5090-38557E   None  None       None
460006  cd04d56d7eaee6c17d749ed6658f8cf3  6e76695156df2f7e  41129a1487edc7a9    1  executing api route (pages) /api/batch-reconci...  ...             7D229463BE07C1C3-38A49E   None  None       None
460005  cd04d56d7eaee6c17d749ed6658f8cf3  595586c3641bacd7  6e76695156df2f7e    3                                               POST  ...             7D229463BE07C1C3-38A49E   None  None       None
460000  cd04d56d7eaee6c17d749ed6658f8cf3  f7abf595bb224451  6e76695156df2f7e    1                                        tcp.connect  ...             F5B9811ED13C5090-38557E   None  None       None
460008  cd04d56d7eaee6c17d749ed6658f8cf3  3916322fef48e6bf  595586c3641bacd7    2             /api/v1/inventory/batch-reconciliation  ...             3970FC721B8AA55E-382DF5   None  None  10.0.2.36

[29714 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Analyze parent-child relationships to identify the most downstream faulty service
# Focus on traces involving the `inventory` service and other faulty components

# Filter traces involving the `inventory` service and other faulty components
traces_faulty_components_filtered = traces_faulty_components[
    traces_faulty_components['serviceName'].str.contains('|'.join(faulty_components), case=False, na=False)
]

# Group by traceId to analyze the call chain for each trace
call_chains = traces_faulty_components_filtered.groupby('traceId')

# Identify the most downstream faulty service by finding spans with no children
# A span with no children will not appear as a parentSpanId in the dataset
downstream_faulty_services = traces_faulty_components_filtered[
    ~traces_faulty_components_filtered['spanId'].isin(traces_faulty_components_filtered['parentSpanId'])
]

downstream_faulty_services

```
Out[6]:
```
The analysis of parent-child relationships in the trace data identified 8,541 spans as the most downstream faulty services in the call chain. These spans include operations such as `SELECT inventory.inventory`, `HikariDataSource.getConnection`, and `/api/v1/inventory/batch-reconciliation`, primarily involving the `inventory` service. This indicates that the `inventory` service is the most downstream faulty service in the call chain during the alert time window. For a detailed view, you can request to display more rows of the data.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                                spanName  ...  __topic__      __tag__:__pack_id__ events links           ip
400182  8eba42c7996a5286267026d41a36f240  b1f5bad4d5f75b7d  23cb4762cdd30c37    3              SELECT inventory.inventory  ...             F0E2F9BF2D649D76-388C92   None  None  10.0.16.157
400184  8eba42c7996a5286267026d41a36f240  ae81fb7082f6866f  23cb4762cdd30c37    1          HikariDataSource.getConnection  ...             F0E2F9BF2D649D76-388C92   None  None  10.0.16.157
400200  8f91542c2c5df59ba65cd3c5160c6eed  44d0f30cefc1d48f  870f08ce95b88bfd    3              SELECT inventory.inventory  ...             89C0234368B18E62-38C45B   None  None    10.0.2.36
400201  2eea3445af5e51af90ee2a34198344c3  6ce9a6c781e56903  68dadfbedcacd398    3              SELECT inventory.inventory  ...             89C0234368B18E62-38C45B   None  None    10.0.2.36
400207  2eea3445af5e51af90ee2a34198344c3  141a53f10cc5ab9b  68dadfbedcacd398    1          HikariDataSource.getConnection  ...             89C0234368B18E62-38C45B   None  None    10.0.2.36
...                                  ...               ...               ...  ...                                     ...  ...        ...                      ...    ...   ...          ...
459998  acd1ba6afdb239440203ac5202e8fd47  8579deea556a370f  4bbe106900e8cedc    1                             tcp.connect  ...             383BF94C620EA31F-3846DE   None  None         None
459999  2801904fa0aaf270cbefb1ddf4dc4220  3ddd42d1a36f2bf5  54a71e47755abc6d    1                             tcp.connect  ...             383BF94C620EA31F-3846DE   None  None         None
460000  cd04d56d7eaee6c17d749ed6658f8cf3  f7abf595bb224451  6e76695156df2f7e    1                             tcp.connect  ...             F5B9811ED13C5090-38557E   None  None         None
460001  cd04d56d7eaee6c17d749ed6658f8cf3  e7f1e15618e10088  6e76695156df2f7e    3                              dns.lookup  ...             F5B9811ED13C5090-38557E   None  None         None
460008  cd04d56d7eaee6c17d749ed6658f8cf3  3916322fef48e6bf  595586c3641bacd7    2  /api/v1/inventory/batch-reconciliation  ...             3970FC721B8AA55E-382DF5   None  None    10.0.2.36

[8541 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert '_time_' column to numeric and filter logs within the alert time window
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

logs_within_window = logs_data[
    (logs_data['_time_'] >= alert_start_microseconds) &
    (logs_data['_time_'] <= alert_end_microseconds)
]

# Filter logs related to the 'inventory' service
inventory_logs = logs_within_window[
    logs_within_window['_container_name_'].str.contains('inventory', case=False, na=False)
]

# Search for error messages or relevant information
inventory_error_logs = inventory_logs[
    inventory_logs['content'].str.contains('error|exception|timeout|fail', case=False, na=False)
]

inventory_error_logs

```
Out[7]:
```
No error messages or relevant logs related to the `inventory` service were found within the alert time window (16:01:41 to 16:06:48, UTC+8). This suggests that the root cause reason for the fault may not be directly observable in the logs. Further investigation using other telemetry data, such as metrics or traces, may be required.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```